# Forecast pipelines

This notebook illustrates how `aifs-modal`'s ingestion and forecast functions compose into pipelines for common use cases.

There are two broad types of forecasting pipelines for `aifs-modal`, namely

| | Implicit ingestion | Explicit ingestion |
|---|---|---|
| **Co-located Modal** | `run_forecast` dispatches `ingest_ifs_arraylake` / `ingest_era5_arco` automatically if ICs are missing on the IC Volume | Call the Modal ingest function first; `run_forecast` skips ingestion |
| **Inline** | `run_forecast` ingests inline (`ifs-ekd`, `era5-cds`) | — |

The three pipelines below cover the most common patterns:

1. **Operational** — near-real-time, one call, fully implicit.
2. **ERA5 reforecast** — batch-ingest a date range first, then loop over forecasts.
3. **Patched-IC scenario** — run a baseline forecast with `keep_ics=True`, modify the ICs, run the patched forecast.

In [ ]:
import datetime as dt

import aifs_modal
from aifs_modal import app, run_forecast

storage_bucket = "aifs-modal-unibe"

# Brightband source repo (ECMWF IFS initial conditions on the ArrayLake marketplace)
ifs_source_repo = "martibosch/ecmwf-ifs-hres-ics-open"

_today_00z = dt.datetime.now(dt.UTC).replace(hour=0, minute=0, second=0, microsecond=0)

# IFS: 2 days ago — safely within the ~15-day Brightband rolling window
ifs_date = _today_00z - dt.timedelta(days=2)

# ERA5: 7 days ago — past the ~5-day publication lag
era5_date = _today_00z - dt.timedelta(days=7)

lead_time = 96  # hours

print(f"IFS  date : {ifs_date.isoformat()}")
print(f"ERA5 date : {era5_date.isoformat()}")

IFS  date : 2026-05-02T00:00:00+00:00
ERA5 date : 2026-04-27T00:00:00+00:00


## Pipeline 1: Operational (near-real-time, fully implicit)

The simplest pipeline. A single `run_forecast` call handles everything:

1. If ICs are missing on the Modal IC Volume, `run_forecast` dispatches
   `ingest_ifs_arraylake` to Modal `us-east` (co-located with the Cloudflare
   R2 ENAM bucket) and waits for it to finish.
2. Once ICs are committed to the Volume, inference runs on a GPU container.
3. ICs are automatically deleted from the Volume after successful inference
   (default `keep_ics=False`).

```
run_forecast (CPU orchestrator)
 ├── ICs missing? → ingest_ifs_arraylake (CPU, us-east) → IC Volume
 └── run_inference (GPU)
```

`run_forecast` is **idempotent**: it checks for an existing output before
ingesting ICs or starting inference, so re-running the cell for the same date
is safe and cheap. `forecast_exists` lets you short-circuit even earlier —
before the ephemeral app is spun up — which matters most in loops or scheduled
pipelines.

In [ ]:
if not aifs_modal.forecast_exists(
    ifs_date, storage_bucket, outputs_prefix="pipelines-p1-outputs"
):
    with app.run():
        run_forecast.remote(
            ifs_date,
            storage_bucket,
            source_repo=ifs_source_repo,
            lead_time=lead_time,
            outputs_prefix="pipelines-p1-outputs",
        )

## Pipeline 2: ERA5 reforecast (explicit batch ingestion)

For hindcasts over a date range — typical when the target dates fall outside
the Brightband rolling window — it is more efficient to decouple ingestion from
inference:

1. **Ingest once**: `ingest_era5_arco` runs in Modal `us-central1`, fetches all
   dates in the range from ARCO-ERA5, and commits in a single transaction.
2. **Forecast loop**: `run_forecast` for each initialisation date; ingestion is
   skipped because the ICs are already committed.

```
ingest_era5_arco (CPU, us-central1)   ← runs once for the full range
  └── commits IC range to icechunk

for each date:
  run_forecast (CPU orchestrator)
    └── ICs present → run_inference (GPU)
```

Separating the two steps also lets you schedule them independently — for
example, ingest a week's worth of ERA5 ICs overnight and run the forecasts
on demand.

In [ ]:
# reforecast window: 4 initialisation dates (one per day at 00z)
era5_end = era5_date.replace(hour=0)
era5_start = era5_end - dt.timedelta(days=3)

# step 1: batch-ingest the full range in us-central1 onto the Modal IC Volume
# ingest_era5_arco takes start/end as 6-hourly bounds;
# we pass the full window so all needed 6h steps are covered
with app.run():
    aifs_modal.ingest_era5_arco.remote(
        era5_start.isoformat(),
        era5_end.isoformat(),
    )

In [ ]:
# step 2: forecast for each 00z initialisation date — skip any already done
# ingestion is skipped — ICs are already on the Volume from step 1
init_dates = [era5_start + dt.timedelta(days=i) for i in range(4)]
pending = [
    d
    for d in init_dates
    if not aifs_modal.forecast_exists(
        d, storage_bucket, outputs_prefix="pipelines-p2-outputs"
    )
]

if pending:
    with app.run():
        for date in pending:
            run_forecast.remote(
                date,
                storage_bucket,
                source="era5-arco",
                lead_time=lead_time,
                outputs_prefix="pipelines-p2-outputs",
            )
            print(f"dispatched forecast for {date.isoformat()}")

## Pipeline 3: Patched-IC scenario (ArrayLake write)

Some experiments modify the initial conditions before running a forecast — for example, patching sea-surface temperatures to mimic a climate-change scenario (see the [CMIP6 SST-patch notebook](cmip6-sst-patch.ipynb)). The recommended approach is to write a custom ArrayLake repository with the patched ICs, then point `run_forecast` at it as IC source:

1. `run_forecast` — run the baseline forecast from the Brightband source.
2. Read two IC dates from Brightband, apply the patch, write dynamic + static fields to a new ArrayLake repo on `main`.
3. `run_forecast(..., source_repo=patched_ics_repo, source_static_branch="main")` — run the patched forecast; `ingest_ifs_arraylake` reads from the patched repo.

```
run_forecast (source_repo=ifs_source_repo)  ← baseline

patch_ics (local) → write to ArrayLake       ← modify skt, write to patched_ics_repo

run_forecast (source_repo=patched_ics_repo, source_static_branch="main")  ← patched
  └── ingest_ifs_arraylake → run_inference (GPU)
```

This approach is independent of `aifs-modal` internals — the patched ICs are versioned ArrayLake commits, reusable without re-running the patch step. For ERA5/historical dates outside the Brightband window, patching on the Modal Volume with `keep_ics=True` is an alternative — see [CMIP6 SST-patch Variations](cmip6-sst-patch.ipynb).

In [ ]:
patched_ics_repo = (
    "martibosch/aifs-modal-sst-patched"  # your own ArrayLake repo for patched ICs
)

# Step 1: run baseline forecast from the Brightband IFS source
if not aifs_modal.forecast_exists(
    ifs_date, storage_bucket, outputs_prefix="pipelines-p3-baseline-outputs"
):
    with app.run():
        run_forecast.remote(
            ifs_date,
            storage_bucket,
            source="ifs-arraylake",
            source_repo=ifs_source_repo,
            lead_time=lead_time,
            outputs_prefix="pipelines-p3-baseline-outputs",
        )

# Step 2: read ICs from Brightband, patch, write to ArrayLake repo (see
# cmip6-sst-patch.ipynb)
# import arraylake, numpy as np, xarray as xr
# client = arraylake.Client(token=os.environ["ARRAYLAKE_API_TOKEN"])
# ... (apply anomaly to skt, write dynamic + static to patched_ics_repo main branch)

# Step 3: run patched forecast — ingest_ifs_arraylake reads from the patched repo
# with app.run():
#     run_forecast.remote(
#         ifs_date,
#         storage_bucket,
#         source="ifs-arraylake",
#         source_repo=patched_ics_repo,
#         source_static_branch="main",
#         lead_time=lead_time,
#         outputs_prefix="pipelines-p3-patched-outputs",
#     )

## Cleanup

Remove all forecast data from the bucket.

In [ ]:
import storage_utils

storage_utils.delete_prefixes(
    storage_bucket,
    "pipelines-p1-outputs",
    "pipelines-p2-outputs",
    "pipelines-p3-baseline-outputs",
    "pipelines-p3-patched-outputs",
)